# Point-in-Time Recovery (PITR)

---
**The Challenge: "Code Red" Scenario**

The Spring Sale went live. During the final deployment, a DevOps engineer intended to clean up a temp staging table:


```sql
DROP TABLE temp_inventory_buffer;  -- intended
DROP TABLE inventory_main;         -- what actually ran
```

**Impact:** Production website throws 500 errors. Customers can't check stock or complete purchases. Revenue loss: thousands per minute.

**Traditional Recovery:** Find backup > Provision instance > Restore (hours) > Replay logs > Validate

**The Lakebase Solution:Point In Time Restore:** 
Restore Production back to a state / timestamp before the accidental DROP operation was executed. A new branch with the desired data is created in seconds.

---

**In this notebook:**
1. Create `inventory_main` table and load data
2. Simulate the accidental DROP TABLE
3. Observe the error
5. Use PITR to create a recovery branch
6. Verify the data is fully restored

### Install Dependencies & Configure Helpers

In [0]:
%pip install databricks-sdk --upgrade -q
%pip install psycopg2-binary -q

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import time
import psycopg2

w = WorkspaceClient()

# Derive project name from the current user's identity
current_user = w.current_user.me()
db_user = current_user.user_name
username_prefix = db_user.split("@")[0].replace(".", "-")
project_name = f"lakebase-branching-workshop-{username_prefix}"

# List branches — the default 'production' branch should exist
branches = list(w.postgres.list_branches(parent=f"projects/{project_name}"))

print(f"📋 Branches in '{project_name}':")
for b in branches:
    branch_id = b.name.split("/branches/")[-1]
    is_default = "⭐ default" if b.status and b.status.default else ""
    print(f"   • {branch_id} {is_default}")

# Get the production branch (the default one, or fallback to the first)
prod_branch = next(
    (b for b in branches if b.status and b.status.default),
    branches[0]
)
prod_branch_name = prod_branch.name
print(f"\n✅ Production branch: {prod_branch_name}")

## Helper: Connect to Any Branch

This function is used by all scenario notebooks (01–05) to connect to a specific branch.
It handles endpoint discovery, waiting, and OAuth token generation.

In [0]:
def connect_to_branch(branch_id, wait_seconds=300):
    """
    Connect to a Lakebase branch endpoint.
    Automatically creates a compute endpoint if none exists.
    
    Args:
        branch_id: Branch name (e.g. "dev-readonly", "feature-loyalty-tier")
        wait_seconds: Max seconds to wait for endpoint to become ready (default 300)
    
    Returns:
        tuple: (connection, host, endpoint_name)
    """
    from databricks.sdk.service.postgres import Endpoint, EndpointSpec, EndpointType, Duration as Dur

    branch_full = f"projects/{project_name}/branches/{branch_id}"
    
    # Check if an endpoint already exists
    endpoints = list(w.postgres.list_endpoints(parent=branch_full))
    
    if not endpoints:
        # Create a compute endpoint for this branch
        ep_id = f"ep-{branch_id}"
        print(f"🔄 Creating compute endpoint for branch '{branch_id}'...")
        w.postgres.create_endpoint(
            parent=branch_full,
            endpoint=Endpoint(spec=EndpointSpec(
                endpoint_type=EndpointType.ENDPOINT_TYPE_READ_WRITE,
                autoscaling_limit_min_cu=min_cu,
                autoscaling_limit_max_cu=max_cu,
                suspend_timeout_duration=Dur(seconds=suspend_timeout_seconds)
            )),
            endpoint_id=ep_id
        ).wait()
        print(f"   ✅ Compute endpoint created!")
        endpoints = list(w.postgres.list_endpoints(parent=branch_full))
    
    # Wait for the endpoint host to be available
    ep = endpoints[0]
    if not ep.status or not ep.status.hosts or not ep.status.hosts.host:
        print(f"⏳ Waiting for endpoint to become ready...")
        for i in range(wait_seconds // 10):
            time.sleep(10)
            endpoints = list(w.postgres.list_endpoints(parent=branch_full))
            ep = endpoints[0]
            if ep.status and ep.status.hosts and ep.status.hosts.host:
                break
            print(f"   Still waiting... ({(i+1)*10}s)")
    
    if not ep.status or not ep.status.hosts or not ep.status.hosts.host:
        raise Exception(f"Endpoint not ready for branch '{branch_id}' after {wait_seconds}s")
    
    host = ep.status.hosts.host
    
    # Generate OAuth token and connect
    cred = w.postgres.generate_database_credential(endpoint=ep.name)
    branch_conn = psycopg2.connect(
        host=host,
        port=5432,
        dbname="databricks_postgres",
        user=db_user,
        password=cred.token,
        sslmode="require"
    )
    branch_conn.autocommit = True
    
    print(f"✅ Connected to branch '{branch_id}'")
    print(f"   Host: {host}")
    return branch_conn, host, ep.name

def delete_branch_safe(branch_id, max_retries=6, wait_between=30):
    """
    Delete a branch, retrying if the endpoint is still reconciling.
    
    Args:
        branch_id: Branch name (e.g. "dev-readonly")
        max_retries: Max number of retry attempts (default 6)
        wait_between: Seconds to wait between retries (default 30)
    """
    branch_full = f"projects/{project_name}/branches/{branch_id}"
    
    for attempt in range(max_retries):
        try:
            w.postgres.delete_branch(name=branch_full).wait()
            print(f"🗑️ Branch '{branch_id}' deleted.")
            return
        except Exception as e:
            if "reconciliation" in str(e).lower() and attempt < max_retries - 1:
                print(f"   ⏳ Endpoint still reconciling, retrying in {wait_between}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_between)
            else:
                raise

print("🔧 connect_to_branch() and delete_branch_safe() helpers defined.")

---
### The Setup - Creating the Critical Table

The DevOps team creates `inventory_main` in production to track real-time stock levels during the Spring Sale. Every product page queries this table.

In [0]:
# connect to production branch
conn, conn_host, conn_endpoint = connect_to_branch('production')
db_schema = "ecommerce"

In [0]:
with conn.cursor() as cur:
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {db_schema}.inventory_main (
        id              SERIAL PRIMARY KEY,
        product_id      INT             NOT NULL REFERENCES {db_schema}.products(id),
        warehouse       VARCHAR(50)     NOT NULL,
        quantity        INT             NOT NULL DEFAULT 0,
        reserved        INT             NOT NULL DEFAULT 0,
        last_updated    TIMESTAMP       NOT NULL DEFAULT NOW()
    );

        INSERT INTO {db_schema}.inventory_main (product_id, warehouse, quantity, reserved) VALUES
            (1,  'US-EAST',  45,  5),
            (1,  'US-WEST',  30,  2),
            (2,  'US-EAST', 180, 15),
            (2,  'EU-WEST',  95,  8),
            (3,  'US-EAST',  25,  3),
            (4,  'US-EAST',  70, 10),
            (4,  'EU-WEST',  55,  6),
            (5,  'US-EAST', 120, 20),
            (6,  'US-EAST', 200,  0),
            (6,  'APAC',    150,  5),
            (7,  'US-EAST',  90,  0),
            (8,  'US-EAST',  60,  4),
            (9,  'US-EAST', 110,  8),
            (10, 'US-EAST',  40, 10),
            (10, 'EU-WEST',  35,  5),
            (11, 'US-EAST', 100,  0),
            (12, 'US-EAST', 250,  0),
            (13, 'EU-WEST',  80,  3),
            (14, 'EU-WEST',  65,  7),
            (15, 'US-EAST', 130,  0)
        ON CONFLICT DO NOTHING;
    """)

---
### The Disaster

The deployment script runs. A variable substitution error caused `inventory_main` to be passed instead of `temp_inventory_buffer`. We add a short delay to ensure the DROP happens strictly after the safe point.

In [0]:
print("Deployment script running... (waiting 3 seconds)")
time.sleep(3)

print()
print("   [SCRIPT] Connecting to production database...")
print("   [SCRIPT] Dropping temporary staging tables...")
print()

try:
    with conn.cursor() as cur:
        cur.execute(f""" DROP TABLE {db_schema}.inventory_main;
                """)
    print("   [SCRIPT] DROP TABLE inventory_main; -- OK")
except Exception as e:
    print(f"   [SCRIPT] ERROR: {e}")

conn.close()

print()
print("   [SCRIPT] Deployment complete.")
print()
print("=" * 60)
print("  INCIDENT TIMELINE")
print("=" * 60)
print(f"  NOW                         <- DROP TABLE inventory_main executed")
print("  ...  Website begins throwing 500 errors")
print("  ...  On-call alert: 'inventory_main not found'")
print("  ...  Revenue impact: $X,XXX per minute")
print("=" * 60)

### Restore using PITR

Point-in-time restore creates a new branch containing your data from the specified point in time. The new branch contains a complete copy of all data and schema as they existed at the restore point.


To perform a restore operation:

1. Open your project  →  Backup & Restore
2. Find the desired snapshot in the list
3. Click "Restore"  →  Confirm


![reset_to_parent_schema_diff.png](Includes/images/branching/lab4.1backup&restore.png)

### New Branch Created

The new branch contains a complete copy of all data and schema as they existed at the restore point. Your original branch remains unchanged and continues operating normally with all its current data.

![reset_to_parent_schema_diff.png](Includes/images/branching/lab4.1newbranch.png)

The new branch is created without an primary compute endpoint attached to it.

![reset_to_parent_schema_diff.png](Includes/images/branching/lab4.1backup&restorebranch_nocompute.png)

We will need to add a primary compute endpoint to the newly created branch

![reset_to_parent_schema_diff.png](/Workspace/Users/benjamin.nwokeleme@databricks.com/LAKEBASE-WORKSHOP/lakebase-02-autoscaling/Includes/images/branching/lab4.1backup&restorebranch_addcompute.png)

### What Happens After New Recovery Branch is created?


Point-in-time restore does not affect existing connections to your original branch. Since the restore creates a new branch rather than modifying your existing branch, all applications and connections continue to work with your original branch without any interruption. To use the restored data, **you must manually update your application connection configuration to point to the new branch.**

You can also explore performing a Back up & Restore (using pg_dump) to move data from the recovered branch to the current production branch

## 🎯 Summary (Update!)

| Step | What Happened |
|---|---|
| **Production branch** | Accidentally deleted important table |
| **Website** | Started throwing errors |
| **PITR** |  Create new branch with data prior to disaster|